# 5. Множественные сравнения

Семейства тестов определены до расчёта поправок:
- **Подтверждающее семейство** - 21 тест основной выборки по доле завершивших (все эксперименты, кроме 263057).
  Только оно входит в правило решений. Поправка - BH на уровне 0,05, Bonferroni справочно.
- **Исследовательские семейства** - триггерный анализ (14 тестов), выборка без не видящих видео (9) и вторичная
  метрика, число задач среди завершивших (21). Для них p показываю без поправки и с BH внутри своего семейства.
  В правило решений они не входят, что бы ни показали.

Зависимость между тестами. Треть учеников (2 751 из 8 205) участвовала больше чем в одном эксперименте, поэтому
тесты не независимы. BH контролирует FDR при независимых или положительно зависимых тестах. Если ученик, которому
одно условие помогло, чаще завершает и другие задания, зависимость скорее положительная, но доказать это нельзя.
Поэтому справочно добавляю поправку Бенджамини-Йекутиели (BY), которая верна при любой зависимости.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy.stats import false_discovery_control, ttest_ind

from src import config, db, effects, stats

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
con = db.connect()
wide = effects.completion_wide(con)
main = effects.effects(wide[wide["sample"] == "all"], np.random.default_rng(config.SEED))

## Подтверждающее семейство: 21 тест основной выборки

In [2]:
conf = main[main.experiment_id != effects.EXCLUDED].copy()
conf["p_bh"] = stats.bh_adjust(conf.p)
conf["p_bonf"] = stats.bonferroni_adjust(conf.p)
conf["p_by"] = false_discovery_control(conf.p, method="by")
conf[["experiment_id", "diff", "p", "p_bh", "p_bonf", "p_by"]].sort_values("p").round(4)

,experiment_id,diff,p,p_bh,p_bonf,p_by
1,237447,-0.0261,0.0405,0.7208,0.8505,1.0
0,226210,-0.0454,0.0992,0.7208,1.0000,1.0
3,241622,-0.0443,0.1098,0.7208,1.0000,1.0
8,250476,0.0418,0.1373,0.7208,1.0000,1.0
2,241501,-0.0439,0.2009,0.7655,1.0000,1.0
20,377658,-0.0228,0.2402,0.7655,1.0000,1.0
12,259379,-0.0511,0.2552,0.7655,1.0000,1.0
11,256027,-0.0330,0.3046,0.7996,1.0000,1.0
10,256017,0.0287,0.3958,0.8153,1.0000,1.0
4,243393,-0.0159,0.4491,0.8153,1.0000,1.0


In [3]:
print("тестов:", len(conf))
for col, name in [("p", "без поправки"), ("p_bh", "BH"), ("p_bonf", "Bonferroni"), ("p_by", "BY")]:
    print(f"{name}: p < 0,05 -", (conf[col] < 0.05).sum())
print("ожидаемое число p < 0,05 при отсутствии всех эффектов:", round(0.05 * len(conf), 2))

тестов: 21
без поправки: p < 0,05 - 1
BH: p < 0,05 - 0
Bonferroni: p < 0,05 - 0
BY: p < 0,05 - 0
ожидаемое число p < 0,05 при отсутствии всех эффектов: 1.05


## Исследовательские семейства

В правило решений не входят. Поправка BH - внутри каждого семейства.

In [4]:
trig_table = effects.trigger_table(con)
symmetric = trig_table[trig_table.symmetric].experiment_id
rng = np.random.default_rng(config.SEED)
trig = effects.effects(wide[(wide["sample"] == "experienced") & wide.experiment_id.isin(symmetric)], rng)
video = effects.effects(wide[wide["sample"] == "video_viewers"], rng)

done = db.run(con, "groups.sql", "completers")
done = done[done.experiment_id != effects.EXCLUDED]
secondary = pd.DataFrame([
    {"experiment_id": eid, "diff": d[d.grp == "E"].log_count.mean() - d[d.grp == "C"].log_count.mean(),
     "p": ttest_ind(d[d.grp == "E"].log_count, d[d.grp == "C"].log_count, equal_var=False).pvalue}
    for eid, d in done.groupby("experiment_id")
])

families = {"триггерный анализ": trig, "без не видящих видео": video, "вторичная метрика (log10 задач)": secondary}
summary = []
for name, f in families.items():
    f["p_bh"] = stats.bh_adjust(f.p)
    summary.append({"семейство": name, "тестов": len(f), "p < 0,05": (f.p < 0.05).sum(),
                    "p_bh < 0,05": (f.p_bh < 0.05).sum()})
pd.DataFrame(summary)

,семейство,тестов,"p < 0,05","p_bh < 0,05"
0,триггерный анализ,14,1,0
1,без не видящих видео,9,0,0
2,вторичная метрика (log10 задач),21,5,2


In [5]:
secondary.sort_values("p")[["experiment_id", "diff", "p", "p_bh"]].head(8).round(4)

,experiment_id,diff,p,p_bh
2,241501,0.0634,0.0000,0.0007
3,241622,0.0588,0.0011,0.0116
1,237447,0.0437,0.0158,0.1107
12,259379,0.1103,0.0295,0.1548
9,255116,0.0401,0.0382,0.1605
20,377938,-0.0352,0.0656,0.2295
13,263015,-0.0278,0.1348,0.3858
17,293151,0.0251,0.1470,0.3858


Вторичная метрика - p из t-теста Уэлча для разницы средних log10(ProblemCount) среди завершивших. Это та же
оговорка, что на этапе 3: сравниваются завершившие, а условие может менять, кто завершает.

## 293151: эффект среди увидевших условие

Среди увидевших условие +8,4 п.п. (p = 0,040), на всех +1,3 п.п. (p = 0,51). Проверяю три вещи: согласуются ли эти
оценки как один эффект, размытый на всех; переживает ли p = 0,040 поправку внутри своего семейства; сколько нужно
учеников со сработавшим условием, чтобы подтвердить эффект такого размера.

In [6]:
EID = 293151
share = trig_table.set_index("experiment_id").trigger_share[EID]
t = trig.set_index("experiment_id").loc[EID]
m = main.set_index("experiment_id").loc[EID]
print("доля увидевших условие:", round(share, 4))
print("эффект среди увидевших, п.п.:", round(100 * t["diff"], 2), " p =", round(t.p, 4), " p_bh =", round(t.p_bh, 4))
print("размытый на всех (доля x эффект), п.п.:", round(100 * share * t["diff"], 2))
print("на всех, п.п.:", round(100 * m["diff"], 2), " интервал", round(100 * m.boot_lo, 1), "...", round(100 * m.boot_hi, 1))

доля увидевших условие: 0.3257
эффект среди увидевших, п.п.: 8.4  p = 0.0402  p_bh = 0.4884
размытый на всех (доля x эффект), п.п.: 2.74
на всех, п.п.: 1.25  интервал -2.4 ... 5.0


Если на не увидевших условие оно не влияет, эффект на всех равен размытому. Разница между ними - это эффект среди
не увидевших, умноженный на их долю. Считаю его напрямую: не увидевшие = все минус увидевшие.

In [7]:
w = wide[wide.experiment_id == EID].set_index("sample")
ne = w.loc["all", ["n_c", "n_e", "done_c", "done_e"]] - w.loc["experienced", ["n_c", "n_e", "done_c", "done_e"]]
z, p = stats.two_prop_ztest(ne.done_e, ne.n_e, ne.done_c, ne.n_c)
lo, hi = stats.diff_ci(ne.done_e, ne.n_e, ne.done_c, ne.n_c)
print("не увидевшие условие: C", ne.n_c, "E", ne.n_e,
      " доли", round(ne.done_c / ne.n_c, 4), round(ne.done_e / ne.n_e, 4))
print("эффект среди не увидевших, п.п.:", round(100 * (ne.done_e / ne.n_e - ne.done_c / ne.n_c), 2),
      " интервал Вальда", round(100 * lo, 1), "...", round(100 * hi, 1), " p =", round(p, 3))

не увидевшие условие: C 433 E 416  доли 0.9238 0.899
эффект среди не увидевших, п.п.: -2.47  интервал Вальда -6.3 ... 1.3  p = 0.204


In [8]:
base = w.loc["experienced", "done_c"] / w.loc["experienced", "n_c"]
rows = []
for label, d in [("как в тесте, +8,4 п.п.", t["diff"]), ("вдвое меньше, +4,2 п.п.", t["diff"] / 2)]:
    n = np.ceil(stats.n_two_prop(base, d))
    rows.append({"эффект": label, "увидевших условие в группе": n, "всего увидевших": 2 * n,
                 "всего учеников при доле увидевших 33%": np.ceil(2 * n / share)})
print("доля завершивших среди увидевших условие в контроле:", round(base, 4))
pd.DataFrame(rows)

доля завершивших среди увидевших условие в контроле: 0.7407


,эффект,увидевших условие в группе,всего увидевших,всего учеников при доле увидевших 33%
0,"как в тесте, +8,4 п.п.",378.0,756.0,2322.0
1,"вдвое меньше, +4,2 п.п.",1614.0,3228.0,9913.0


## Итог

- Подтверждающее семейство. Без поправки p < 0,05 один тест из 21 (237447, -2,6 п.п.). При 21 тесте без единого
  эффекта случайно ожидается 1,05 такого результата. После BH, Bonferroni и BY значимых тестов 0. Минимальный p после
  BH - 0,72.
- Исследовательские семейства. Триггерный анализ: одно p < 0,05 из 14, после BH внутри семейства 0. Без не видящих
  видео: 0 из 9. Вторичная метрика: 5 из 21 без поправки, после BH 2 - 241501 и 241622, в обоих у завершивших в E
  больше задач (на 16% и 14% по геометрическому среднему). Это наблюдение, а не решение: отбор завершивших.
- 293151 согласуется с одним эффектом, размытым на всех. Доля увидевших условие 0,326, 0,326 x 8,4 = 2,7 п.п.
  против 1,3 п.п. на всех, и 2,7 лежит внутри интервала для всех (от -2,4 до +5,0). Разница объясняется не увидевшими
  условие: среди них -2,5 п.п., интервал от -6,3 до +1,3, p = 0,20, то есть совместимо с нулём.
- p = 0,040 поправку внутри семейства из 14 триггерных тестов не переживает: после BH 0,49. В таблице решений
  293151 - «мало данных» по правилу, с пометкой «гипотеза для повторного теста».
- Чтобы подтвердить +8,4 п.п. среди увидевших условие при α = 0,05 и мощности 0,8, нужно 378 таких учеников в группе,
  756 всего, это около 2 300 учеников при доле увидевших 33%. В тесте увидевших было 410. Эффект, отобранный по
  p < 0,05, обычно завышен (проклятие победителя), поэтому реалистичнее планировать на вдвое меньший: +4,2 п.п.
  требуют 1 614 увидевших в группе, около 9 900 учеников.